# IEE 575 — Lab 1: Into the Gaussian World

**Name:** &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **Date:**

---

This lab builds the foundation for Gaussian Processes.  
By the end you will understand:
- Why Gaussians are the right tool for modelling uncertainty
- What marginalization and conditioning mean geometrically
- How the noisy-sensor problem motivates GP regression

> **AI rule:** You may ask AI *"explain why this error occurs"* or *"what does this parameter do?"*  
> You may **not** ask AI to write or fix your code. If you use AI output, be ready to explain any line on demand.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import multivariate_normal, norm

rng = np.random.default_rng(12345)

---
## Section 1 — Motivation: The Noisy Sensor

> **The problem that drives this entire lab.**

We throw a ball upward. Its **true** height at time $t$ is known to us (for now):

$$h(t) = 20t - \frac{1}{2} \times 9.8 \times t^2$$

A sensor measures the ball's height at a few random moments — but every reading has noise.  
Our goal: *reconstruct the smooth curve from the noisy dots.*

---

### Task 1.1 — Implement and plot the true trajectory

**(a)** Write a function `height(t)` for the formula above.  
**(b)** Create `t_true` — 200 evenly-spaced points from 0 to 4 s using `np.linspace`.  
**(c)** Compute `h_true` by applying `height` directly to `t_true` — **no loop**.  
**(d)** Plot the true curve. Add a dashed grey line at $h = 0$ with `plt.axhline`.  
**(e)** Use `np.argmax` to find and mark the peak with a red dot.

**PREDICT** — what shape do you expect the curve to have?
> *Your answer...*

In [ ]:
# (a) ---- height function ----


# (b, c) ---- time array and true heights ----


# (d, e) ---- plot true curve, ground line, and peak ----


plt.xlabel("Time (s)")
plt.ylabel("Height (m)")
plt.title("True Trajectory")
plt.legend()
plt.grid(True)
plt.show()

### Task 1.2 — Add the noisy sensor

The sensor fires at 12 random moments. Each reading has Gaussian noise with $\sigma = 1.5$ m.

**(a)** The sensor observations are already set up below — read the code carefully before running.  
**(b)** Scatter-plot the noisy observations **on top of** your true curve from Task 1.1.  
**(c)** Add a shaded uncertainty band using `plt.fill_between`:
```python
plt.fill_between(t_true, h_true - 1.5, h_true + 1.5,
                 alpha=0.2, color='steelblue', label='±1σ sensor noise')
```

**PREDICT** — will all noisy dots fall inside the shaded band? Why or why not?
> *Your answer...*

In [ ]:
# Sensor observations — read carefully, then plot on top of Task 1.1
np.random.seed(42)
t_obs = np.random.uniform(0, 4, size=12)
noise = np.random.normal(0, 1.5, size=12)
h_obs = height(t_obs) + noise        # apply your function here

# ---- re-plot true curve ----


# ---- scatter noisy observations ----


# ---- add uncertainty band ----


plt.xlabel("Time (s)")
plt.ylabel("Height (m)")
plt.title("True Trajectory vs Noisy Sensor Readings")
plt.legend()
plt.grid(True)
plt.show()

### Reflect

> If you did **not** know the formula for `height(t)` — you only had the noisy dots —  
> how would you reconstruct the smooth curve?  
> Write your intuition here. We will answer this properly when we study Gaussian Processes.

> *Your answer...*

---

---
## Section 2 — Univariate Gaussian Distribution

A random variable $X \sim \mathcal{N}(\mu, \sigma^2)$ has probability density:

$$f(x) = \frac{1}{\sqrt{2\pi}\,\sigma} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

Two parameters fully define it:
- **Mean** $\mu$ — locates the peak; $f(x)$ is maximised when $x = \mu$
- **Variance** $\sigma^2$ — controls the spread around that peak

> **Compare with Section 1:** the noise term `np.random.normal(0, 1.5)` drew samples  
> from exactly this distribution with $\mu=0$, $\sigma=1.5$.

---

### Task 2.1 — Implement and plot the density

The function below implements $f(x)$ directly (without using `scipy`).  
Study it, then complete the plotting loop.

**PREDICT** — among the four parameter sets, which curve will be tallest? Widest?
> *Your answer...*

In [ ]:
def univariate_density(x, mu, sigma):
    """Univariate Gaussian PDF evaluated at x."""
    return (2 * np.pi * sigma**2)**(-0.5) * np.exp(-0.5 * ((x - mu) / sigma)**2)

x_vals = np.linspace(-10, 10, 1000)
params = [(-3, 1), (0, 1), (0, 5), (1, 0.5)]   # (mu, sigma) pairs

for mu, sigma in params:
    p_vals = univariate_density(x_vals, mu, sigma)
    plt.plot(x_vals, p_vals, label=rf"$\mu={mu},\; \sigma={sigma}$")

plt.xlabel(r"$x$")
plt.ylabel(r"$f(x)$")
plt.title("Univariate Gaussian — effect of $\mu$ and $\sigma$")
plt.legend()
plt.grid(True)
plt.show()

### Task 2.2 — Connect to the sensor noise

The noise in Task 1.2 was `np.random.normal(0, 1.5)`.

**(a)** Add a fifth entry `(0, 1.5)` to the `params` list above and re-run.  
**(b)** Which curve corresponds to the sensor noise? Mark it by changing its `linewidth` or `linestyle`.  
**(c)** **PREDICT:** what fraction of noise samples should fall within $\pm 1.5$ m of zero?  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Check numerically:

```python
np.mean(np.abs(noise) < 1.5)
```

> *Your prediction and result...*

---

---
## Section 3 — Multivariate Gaussian Distribution

A $k$-dimensional random vector $\mathbf{X} = (X_1, \ldots, X_k)^\top \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)$ has density:

$$f(\mathbf{x}) = \frac{1}{(2\pi)^{k/2}|\Sigma|^{1/2}}
\exp\!\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

Two parameters:
- **Mean vector** $\boldsymbol{\mu} = (E[X_1], \ldots, E[X_k])^\top$
- **Covariance matrix** $\Sigma_{ij} = \operatorname{Cov}(X_i, X_j)$ — symmetric, positive semi-definite

> **The scalar $(x-\mu)^2/\sigma^2$ generalises to the Mahalanobis distance**  
> $(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})$

---

### The Covariance Matrix

$$\Sigma_{k\times k} = \begin{bmatrix}
\sigma_1^2 & \operatorname{cov}(x_1,x_2) & \cdots & \operatorname{cov}(x_1,x_k) \\
\operatorname{cov}(x_2,x_1) & \sigma_2^2 & \cdots & \operatorname{cov}(x_2,x_k) \\
\vdots & \vdots & \ddots & \vdots \\
\operatorname{cov}(x_k,x_1) & \operatorname{cov}(x_k,x_2) & \cdots & \sigma_k^2
\end{bmatrix}$$

- **Diagonal** — variances $\sigma_i^2$ of each variable  
- **Off-diagonal** — pairwise covariances; $\Sigma$ is symmetric so $\operatorname{cov}(x_i,x_j) = \operatorname{cov}(x_j,x_i)$

> **Question:** what happens when all off-diagonal entries are zero?  
> What does that say about the variables?
> *Your answer...*

### Task 3.1 — Plot bivariate contours

The helper function below computes the PDF on a grid.  
Run the three examples and answer the questions after each one.

In [ ]:
def compute_bivariate_pdf(mean, cov, num_points=100):
    """Return meshgrid (Y1, Y2) and PDF values for a 2D Gaussian."""
    cov = np.array(cov)
    y1 = np.linspace(mean[0] - 3*np.sqrt(cov[0,0]),
                     mean[0] + 3*np.sqrt(cov[0,0]), num=num_points)
    y2 = np.linspace(mean[1] - 3*np.sqrt(cov[1,1]),
                     mean[1] + 3*np.sqrt(cov[1,1]), num=num_points)
    Y1, Y2 = np.meshgrid(y1, y2)
    Y = np.column_stack([Y1.ravel(), Y2.ravel()])
    pdf = multivariate_normal.pdf(Y, mean=mean, cov=cov).reshape(num_points, num_points)
    return Y1, Y2, pdf

def plot_bivariate(mean, cov, title):
    Y1, Y2, pdf = compute_bivariate_pdf(mean, cov)
    plt.figure(figsize=(4, 4))
    plt.contourf(Y1, Y2, pdf, cmap='viridis')
    plt.colorbar(label='density')
    plt.xlabel(r'$X_1$');  plt.ylabel(r'$X_2$')
    plt.title(title);  plt.axis('equal');  plt.tight_layout();  plt.show()

In [ ]:
# Example A — zero covariance (identity matrix)
plot_bivariate(mean=[0, 0],
               cov=[[1, 0.0], [0.0, 1]],
               title=r"$\Sigma = I$ — zero covariance")

**What shape are the contours? Why?**
> *Your answer...*

In [ ]:
# Example B — positive covariance
plot_bivariate(mean=[0, 0],
               cov=[[1, 0.7], [0.7, 1]],
               title=r"Positive covariance $(0.7)$")

**In which direction does the ellipse tilt? Why?**
> *Your answer...*

In [ ]:
# Example C — negative covariance
plot_bivariate(mean=[0, 0],
               cov=[[1, -0.7], [-0.7, 1]],
               title=r"Negative covariance $(-0.7)$")

**TASK:** try changing the diagonal entries to `[[2, 0.7], [0.7, 0.5]]`.  
How does unequal variance change the shape?
> *Your answer...*

---

---
## Section 4 — Marginalization

Given a joint distribution, we recover the distribution of a single variable by integrating out the rest:

$$f(X_1) = \int f(X_1, X_2)\, dX_2$$

**Key result:** every marginal of a Gaussian is itself Gaussian:

$$X_i \sim \mathcal{N}(\mu_i,\; \Sigma_{ii})$$

The marginal only depends on the diagonal entry of $\Sigma$ — the off-diagonal covariance disappears.

---

### Task 4.1 — Visualise marginalisation

The cell below plots the joint distribution in 3D and projects the marginals onto the side walls.  
Run it, then answer the questions below.

In [ ]:
mean  = [0, 0]
cov   = np.array([[2, 0.9], [0.9, 0.5]])

num_points = 100
mvn = multivariate_normal(mean=mean, cov=cov)

# Sample cloud
samples = mvn.rvs(num_points**2, random_state=42)

# Marginal grids
y1 = np.linspace(mean[0] - 3*np.sqrt(cov[0,0]),
                 mean[0] + 3*np.sqrt(cov[0,0]), num=num_points)
y2 = np.linspace(mean[1] - 3*np.sqrt(cov[1,1]),
                 mean[1] + 3*np.sqrt(cov[1,1]), num=num_points)
Y1, Y2 = np.meshgrid(y1, y2)

pdf_y1 = norm.pdf(Y1.ravel(), loc=mean[0], scale=np.sqrt(cov[0,0])).reshape(num_points, num_points)
pdf_y2 = norm.pdf(Y2.ravel(), loc=mean[1], scale=np.sqrt(cov[1,1])).reshape(num_points, num_points)

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(projection='3d')

ax.scatter(samples[:,0], samples[:,1], np.zeros(len(samples)), s=0.5, alpha=0.3, label='joint samples')
ax.scatter(Y1,  6*np.ones((num_points, num_points)), pdf_y1, s=1, color='steelblue',  label=r'marginal $X_1$')
ax.scatter(-6*np.ones((num_points, num_points)), Y2, pdf_y2, s=1, color='darkorange', label=r'marginal $X_2$')

ax.set_xlabel(r'$X_1$');  ax.set_ylabel(r'$X_2$');  ax.set_zlabel('density')
ax.set_xlim(-6, 6);  ax.set_ylim(-6, 6)
ax.set_title('Joint distribution and its marginals')
plt.tight_layout();  plt.show()

**Questions:**

(a) The joint samples form an elongated cloud — why?  
> *Your answer...*

(b) The two marginal curves are Gaussians. What are their means and variances?  
&nbsp;&nbsp;&nbsp;&nbsp;Read them off from `mean` and `cov` above, then verify with `np.mean(samples[:,0])` etc.  
> *Your answer...*

(c) **TASK:** change `cov[0,1]` from `0.9` to `0.0` and re-run.  
&nbsp;&nbsp;&nbsp;&nbsp;Do the marginals change? Should they?  
> *Your answer...*

---

---
## Section 5 — Conditioning

If we observe $X_1 = x_1$, what is the distribution of $X_2$?

**Key result:** the conditional is also Gaussian:

$$X_2 \mid (X_1 = x_1) \;\sim\; \mathcal{N}(\mu_*, \sigma^2_*)$$

where:

$$\mu_* = \mu_2 + \frac{\Sigma_{21}}{\Sigma_{11}}(x_1 - \mu_1)
\qquad
\sigma^2_* = \Sigma_{22} - \frac{\Sigma_{21}^2}{\Sigma_{11}}$$

> **This is the heart of GP regression.** Conditioning on observations updates both the  
> predicted mean *and* the uncertainty — exactly what you saw in the noisy sensor plot.

---

### Task 5.1 — Implement the conditional formulas

In [ ]:
def conditional_gaussian(mu1, mu2, sig11, sig12, sig22, x1_observed):
    """
    Return (mu_star, var_star) for X2 | X1 = x1_observed.

    Parameters
    ----------
    mu1, mu2     : marginal means
    sig11        : Var(X1)
    sig12        : Cov(X1, X2)
    sig22        : Var(X2)
    x1_observed  : the observed value of X1
    """
    mu_star  = mu2 + (sig12 / sig11) * (x1_observed - mu1)
    var_star = sig22 - (sig12**2 / sig11)
    return mu_star, var_star

# ---- test ----
mu_s, var_s = conditional_gaussian(0, 0, 1.0, 0.9, 1.0, x1_observed=1.0)
print(f"mu*  = {mu_s:.4f}")
print(f"var* = {var_s:.4f}")
print(f"std* = {np.sqrt(var_s):.4f}")

**PREDICT** — before running:  
- Should `mu*` be positive or negative when `x1_observed = 1.0` and `sig12 = 0.9`?  
- Should `var*` be larger or smaller than `sig22 = 1.0`?  
> *Your answer...*

### Task 5.2 — Visualise conditioning (2D)

The plot shows a slice through the joint density at a fixed $X_1$ value.  
Run both examples and compare them.

In [ ]:
def plot_conditioning(cov_val, x1_cut=0.8, title=''):
    """Contour plot of joint density with a vertical slice at x1_cut."""
    cov_matrix = np.array([[1, cov_val], [cov_val, 1]])
    Y1, Y2, pdf = compute_bivariate_pdf([0,0], cov_matrix)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Left: contour + slice line
    axes[0].contourf(Y1, Y2, pdf, cmap='viridis', levels=20)
    axes[0].axvline(x1_cut, color='white', linewidth=2, label=f'$X_1 = {x1_cut}$')
    axes[0].set_xlabel(r'$X_1$');  axes[0].set_ylabel(r'$X_2$')
    axes[0].set_title('Joint density')
    axes[0].legend()

    # Right: conditional distribution at x1_cut
    mu_s, var_s = conditional_gaussian(0, 0,
                                       cov_matrix[0,0], cov_matrix[0,1], cov_matrix[1,1],
                                       x1_cut)
    x2_range = np.linspace(-3, 3, 300)
    cond_pdf  = norm.pdf(x2_range, loc=mu_s, scale=np.sqrt(var_s))

    axes[1].plot(x2_range, cond_pdf, color='steelblue', linewidth=2)
    axes[1].axvline(mu_s, color='red', linestyle='--', label=rf'$\mu_*={mu_s:.2f}$')
    axes[1].fill_between(x2_range, cond_pdf, alpha=0.2, color='steelblue')
    axes[1].set_xlabel(r'$X_2$');  axes[1].set_ylabel('density')
    axes[1].set_title(rf'$X_2 \mid X_1={x1_cut}$   ($\sigma_*={np.sqrt(var_s):.2f}$)')
    axes[1].legend()

    plt.suptitle(title, fontsize=13)
    plt.tight_layout();  plt.show()

In [ ]:
# Zero covariance — conditioning tells us nothing
plot_conditioning(cov_val=0.0, title=r"$\Sigma_{12} = 0$ — independent variables")

**What do you notice about the conditional distribution when covariance is zero?**
> *Your answer...*

In [ ]:
# Near-perfect correlation — conditioning collapses uncertainty
plot_conditioning(cov_val=0.95, title=r"$\Sigma_{12} = 0.95$ — strong correlation")

**Questions:**

(a) How does the conditional `std*` compare between the two cases? Why?  
> *Your answer...*

(b) **TASK:** change `x1_cut` to `0.0`, then `-1.0`. How does `mu*` change?  
&nbsp;&nbsp;&nbsp;&nbsp;Does `var*` change? Should it?  
> *Your answer...*

(c) **TASK:** change `cov_val` to `-0.9`. What happens to the sign of `mu*`? Why?  
> *Your answer...*

---

---
## Section 6 — Closing: Back to the Noisy Sensor

You have now built all the pieces.  
Let's connect them back to where we started.

**The noisy sensor problem stated as conditioning:**

- The sensor observations $(t_{\text{obs}}, h_{\text{obs}})$ give us $\mathbf{X}_1 = \mathbf{h}_{\text{obs}}$  
- The query points $t_{\text{query}}$ are $\mathbf{X}_2$ — unobserved  
- We want $p(\mathbf{X}_2 \mid \mathbf{X}_1 = \mathbf{h}_{\text{obs}})$

If we model the **joint distribution** of all heights as a multivariate Gaussian,  
then conditioning gives us a Gaussian prediction at every query point — with uncertainty.

**That is exactly a Gaussian Process.**

---

### Task 6.1 — Revisit the sensor plot with conditioning in mind

In [ ]:
# Two sensor observations
t1_obs, h1_obs = 1.0, height(1.0) + 1.2    # noisy reading at t=1
t2_obs, h2_obs = 2.5, height(2.5) - 0.8    # noisy reading at t=2.5

# Query: what is the height at t=1.75?
t_query = 1.75

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_true, h_true, 'k-', linewidth=2, label='True curve (unknown in real life)')
ax.fill_between(t_true, h_true - 1.5, h_true + 1.5,
                alpha=0.15, color='steelblue', label='±1σ sensor noise')
ax.scatter([t1_obs, t2_obs], [h1_obs, h2_obs],
           color='steelblue', s=80, zorder=5, label='Sensor observations')
ax.axvline(t_query, color='darkorange', linestyle='--', label=f't_query = {t_query}')
ax.set_xlabel('Time (s)');  ax.set_ylabel('Height (m)')
ax.set_title('What is the height at the query point?')
ax.legend();  ax.grid(True)
plt.tight_layout();  plt.show()

print("Lab 2 will answer this using Gaussian Process Regression.")
print("The key: model the JOINT distribution of all heights, then condition on observations.")

---
## 📝 End-of-Lab Reflection

Answer honestly before you leave — your answers shape the next lab.

**1. Which section was hardest? Why?**
> *Your answer...*

**2. In Section 5, `var*` was always smaller than `var(X2)`.  
&nbsp;&nbsp;&nbsp;In words, why does observing $X_1$ reduce our uncertainty about $X_2$?**
> *Your answer...*

**3. Did you use AI today? For what — and did its explanation match your understanding?**
> *Your answer...*

**4. Write one question you still have:**
> *Your question...*

---
*Well done — you've built the mathematical foundation for Gaussian Processes.* 🎉